In [ ]:
import random
import re
from datasets import load_dataset, get_dataset_config_names
from collections import Counter
import time

# =================================================================
# 🏆 Mission Goal: 한국어 텍스트 데이터셋 분석 미니 프로젝트 🏆
# =================================================================

# [데이터셋 정보]
# CocoRoF/cc-100-korean: 코코로프(CocoRoF)에서 구축한 대규모 한국어 텍스트 데이터셋입니다.
# 대략적인 의미: 이 데이터셋은 방대한 양의 한국어 문장들로 이루어져 있어,
# 텍스트 기반의 AI 모델(예: LLM)을 학습시키거나, 한국어 문장 패턴을 분석하는 기초 자료로 사용됩니다.
# 쉽게 말해, '한국어 어휘력과 문장 구조'를 기계에게 가르치는 엄청 큰 참고서와 같습니다!

DATASET_NAME = "CocoRoF/cc-100-korean"
SAMPLE_COUNT = 50  # 분석을 위해 가져올 샘플 개수 (데이터셋이 너무 크므로, 적은 양으로 진행해요!)

print("✨ 안녕하세요, 미래의 AI 엔지니어님! 튜터가 도와드릴게요! ✨")
print(f"오늘 분석할 데이터셋은 '{DATASET_NAME}' 입니다. 텍스트를 분석하며 AI의 원리를 재미있게 배워봅시다!")
print("--------------------------------------------------------------------------")


# 1. 데이터셋 로딩 및 스트리밍 전략 설정
# 스트리밍을 사용해서 메모리 폭발 없이 거대한 데이터셋을 처리하는 게 핵심입니다!

dataset = None
try:
    # 1-1. 먼저 스트리밍 모드로 로드해봅니다. (가장 효율적인 방법!)
    print("[Step 1/4] 🚀 데이터셋을 스트리밍(Streaming) 모드로 로드 시도...")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공! 스트리밍 모드로 데이터셋을 연결했습니다. (메모리 걱정 NO!)")

except Exception as e:
    # 1-2. 만약 스트리밍 로드에 실패하거나 문제가 발생하면,
    # 일반 모드로 아주 적은 샘플만 다운로드하여 진행합니다.
    print(f"⚠️ 경고! 스트리밍 로드 중 오류 발생 ({e}). 일반 로딩으로 대체합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='train[:1000]', streaming=False)
        print("✅ 성공! 일반 모드로 샘플 데이터 1000개를 로드했습니다.")
    except Exception as e_fallback:
        print(f"🚨 치명적 오류: 데이터 로드에 실패했습니다. {e_fallback}")
        exit()


# 2. 샘플링 데이터 추출 (핵심: 전체 데이터셋을 메모리에 올리지 않기!)
# 전체 데이터셋은 너무 크기 때문에, 재미있는 분석을 위해 상위 K개만 샘플링 합니다.

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)이므로 이 패턴 사용
    print(f"\n[Step 2/4] ✨ 상위 {SAMPLE_COUNT}개의 샘플만 추출합니다...")
    # 스트리밍 데이터셋에서 상위 N개를 뽑아내고, 리스트로 변환합니다.
    sampled_dataset_iterator = iter(dataset.take(SAMPLE_COUNT))
    # 실제 처리를 위해 메모리에 로드할 리스트로 변환 (필수)
    sampled_dataset_list = list(sampled_dataset_iterator)
else:
    # 일반 데이터셋 (Dataset)인 경우
    print(f"\n[Step 2/4] ✨ 상위 {SAMPLE_COUNT}개의 샘플을 추출합니다...")
    # 일반 데이터셋의 경우, select를 쓰면 안 되므로 처음 K개를 슬라이싱합니다.
    sampled_dataset_list = list(dataset.select(range(min(SAMPLE_COUNT, len(dataset)))))

print(f"💡 준비 완료! 총 {len(sampled_dataset_list)}개의 샘플을 활용하여 분석을 진행합니다.")


# =================================================================
# 🧠 실습 1: 텍스트 정량적 분석 (Data Analysis)
# =================================================================

print("\n" + "="*60)
print("🌟 [실습 1] 텍스트 기반 정량 분석: 데이터의 '지문' 읽기")
print("="*60)

# 1-1. 데이터 길이 분포 분석
total_chars = 0
text_lengths = []

for sample in sampled_dataset_list:
    # 현재 데이터셋은 'text'라는 하나의 필드를 가집니다.
    text = sample['text']
    length = len(text)
    text_lengths.append(length)
    total_chars += length

average_length = total_chars / len(sampled_dataset_list)
print(f"📋 1. 전체 샘플의 평균 문자열 길이: 약 {average_length:.2f} 자")

# 1-2. 핵심 단어 추출 (간단한 형태소 분석의 대체재)
# 한글은 복잡하지만, 여기서는 공백이나 구두점 기준으로 '단어'를 분리해봅니다.
all_words = []
for sample in sampled_dataset_list:
    text = sample['text']
    # 한글/영문/숫자만 남기고 공백이나 구두점을 기준으로 분리합니다.
    words = re.findall(r'\S+', text) 
    all_words.extend(words)

# 자주 나오는 단어 찾기
word_counts = Counter(all_words)
most_common_words = word_counts.most_common(5)

print("\n🔍 2. 샘플에서 추출된 상위 5개 단어 (자주 등장하는 키워드):")
for word, count in most_common_words:
    print(f"  - '{word}' (등장 횟수: {count}회)")


# =================================================================
# 🤖 실습 2: AI Prompt 생성 시뮬레이션 (Creative Application)
# =================================================================

print("\n" + "="*60)
print("🧠 [실습 2] AI Prompt 생성기: 데이터를 LLM에 먹이는 방법")
print("="*60)

def create_ai_prompt(text_sample: str, prompt_type: str) -> str:
    """
    특정 텍스트 샘플을 가지고, AI가 처리할 수 있는 형식의 프롬프트를 생성합니다.
    (진짜 AI가 데이터를 받기 전에 사람이 프롬프트로 포장하는 단계와 유사합니다.)
    """
    if prompt_type == "Summarization":
        # LLM에게 요약을 요청하는 프롬프트 포맷
        return f"[TASK: Text Summarization]\nINPUT TEXT: \"{text_sample[:100]}...\"\n\n-> 위 텍스트의 핵심 내용을 3줄로 요약해 주세요."
    elif prompt_type == "Keyword_Extraction":
        # LLM에게 키워드 추출을 요청하는 프롬프트 포맷
        return f"[TASK: Keyword Extraction]\nINPUT TEXT: \"{text_sample}\"\n\n-> 위 텍스트에서 가장 중요한 주제어 5개를 리스트로 뽑아주세요."
    else:
        return f"[TASK: Generic Analysis]\nINPUT TEXT: \"{text_sample}\"\n\n-> 이 텍스트에 대해 궁금한 점을 질문해주세요."

print("💡 목적: AI에게 데이터를 그냥 던지는 것이 아니라, '임무(TASK)'를 부여하는 것이 핵심입니다.")
print("🔍 샘플 텍스트 3개를 골라 AI 프롬프트로 변신시켜 보겠습니다!")

# 임의의 샘플 3개 선택
sample_indices = random.sample(range(len(sampled_dataset_list)), 3)
selected_samples = [sampled_dataset_list[i] for i in sample_indices]

# Prompt 1: 요약 요청
sample1_text = selected_samples[0]['text']
prompt1 = create_ai_prompt(sample1_text, "Summarization")
print("\n--- [Task 1] 요약 요청 프롬프트 ---")
print(prompt1)

# Prompt 2: 키워드 추출 요청
sample2_text = selected_samples[1]['text']
prompt2 = create_ai_prompt(sample2_text, "Keyword_Extraction")
print("\n--- [Task 2] 키워드 추출 프롬프트 ---")
print(prompt2)

# Prompt 3: 자유 질문 요청
sample3_text = selected_samples[2]['text']
prompt3 = create_ai_prompt(sample3_text, "Generic Analysis")
print("\n--- [Task 3] 일반 분석 요청 프롬프트 ---")
print(prompt3)


# =================================================================
# 🎉 마무리 & 격려
# =================================================================
print("\n" + "="*60)
print("✨ 축하합니다! AI 데이터 분석 첫걸음을 성공하셨습니다! ✨")
print("💡 배운 점 요약:")
print("1. 스트리밍(streaming=True)을 통해 대용량 데이터를 효율적으로 다룰 수 있다.")
print("2. 데이터를 분석할 때는 '임무(Task)'를 정의하여 프롬프트 형태로 가공해야 한다.")
print("3. 단순히 데이터를 보는 것을 넘어, 어떤 질문을 던질지 고민하는 것이 AI 엔지니어의 가장 중요한 능력입니다!")
print("👏👏👏 아주 훌륭했어요! 코딩 연습을 더 해봐요! 💖")